In [ ]:
import sympy as sym
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import numpy as np
import pandas as pd

# Load in the required datasets
gld_mass_balance_filename = 'Datasets/ice-sheet-mass-balance.csv' #relative filepath
gld_mass_balance_csv = pd.read_csv(gld_mass_balance_filename)

gld_mass_balance_csv["Day"] = pd.to_datetime(gld_mass_balance_csv["Day"])

plt.figure(figsize=(10,5))
plt.plot(gld_mass_balance_csv["Day"], gld_mass_balance_csv["Seasonal variation"])
plt.xlabel("Year"); plt.ylabel("Greenland Ice Sheet Mass Balance Change (billion tonnes)")
plt.show()

# Calibration value: 2001 thickness of around 1600m (Thomas et al, 2001)
calb_year = pd.to_datetime('2017-01-01')
calb_avg_thickness = 1673 #m

# Let the ice cap mass be directly proportional to the height of the ice cap 
# Assumption: Volume = w * l * h, and its a prism with constant cross-sectional area
# So let V be directly prop. to h. 
# V = m * density, so assume constant density also.

rho_ice = 917 #kg/m^3 NOOOOOOOO
sheet_area = 1.7e12 # m^2 ("1.7 million sqare km ")

# Recalibrate this plot to be height change over time. 
date_idx = gld_mass_balance_csv["Day"].sub(calb_year).abs().idxmin()
calb_mass = gld_mass_balance_csv.loc[date_idx, "Seasonal variation"]

mass_kg = gld_mass_balance_csv["Seasonal variation"] * 1e12
delta_V = mass_kg / rho_ice 
delta_h = delta_V / sheet_area

# calb_offset = calb_avg_thickness - calb_mass

delta_h_calb = delta_h - delta_h.iloc[date_idx]
gld_mass_balance_csv["height calibrated"] = calb_avg_thickness + delta_h_calb

t_historical = gld_mass_balance_csv["Day"]
h_historical = gld_mass_balance_csv["height calibrated"]

plt.figure(figsize=(10,5))
plt.plot(t_historical, h_historical)
plt.xlabel("Year"); plt.ylabel("Greenland Ice Sheet Height (Calibrated)")
plt.grid()
plt.show()



"Results from all MAR simulations indicate that (i) the
period 1961–1990, commonly chosen as a stable reference
period for Greenland SMB and ice dynamics, is actually a
period of anomalously positive SMB (∼ +40 Gt yr−1
) compared to 1900–2010; " 

-> Reconstructions of the 1900–2015 Greenland ice sheet surface mass
balance using the regional climate MAR model


In [ ]:
# Newer dataset

gld_mass_balance_filename_v2 = 'Datasets/imbie_greenland_2021_Gt.csv' #relative filepath
gld_mass_balance_csv_v2 = pd.read_csv(gld_mass_balance_filename_v2)

def decimal_year_to_datetime(y):
    year = int(y)
    remainder = y - year
    start = pd.Timestamp(f"{year}-01-01")
    end = pd.Timestamp(f"{year+1}-01-01")
    return start + (end - start) * remainder


gld_mass_balance_csv_v2["Day"] = (
    gld_mass_balance_csv_v2["Year"]
    .astype(float)
    .apply(decimal_year_to_datetime)
)


plt.figure(figsize=(10,5))
plt.plot(gld_mass_balance_csv_v2["Day"], gld_mass_balance_csv_v2["Cumulative mass balance (Gt)"])
plt.xlabel("Year"); plt.ylabel("Greenland Ice Sheet Mass Balance Change (Giga tonnes)")
plt.show()

# Same calibration method 

calb_year = pd.to_datetime('2017-01-01')
calb_avg_thickness = 1673  # m

rho_ice = 917        # kg/m^3
sheet_area = 1.7e12  # m^2

date_idx_v2 = gld_mass_balance_csv_v2["Day"].sub(calb_year).abs().idxmin()
calb_mass_v2 = gld_mass_balance_csv_v2.loc[date_idx_v2, "Cumulative mass balance (Gt)"]
mass_kg_v2 = gld_mass_balance_csv_v2["Cumulative mass balance (Gt)"] * 1e12
delta_V_v2 = mass_kg_v2 / rho_ice
delta_h_v2 = delta_V_v2 / sheet_area
delta_h_calb_v2 = delta_h_v2 - delta_h_v2.iloc[date_idx_v2]

gld_mass_balance_csv_v2["height calibrated"] = (
    calb_avg_thickness + delta_h_calb_v2
)


t_historical = gld_mass_balance_csv_v2["Day"]
h_historical = gld_mass_balance_csv_v2["height calibrated"]

plt.figure(figsize=(10,5))
plt.plot(
    gld_mass_balance_csv_v2["Day"],
    gld_mass_balance_csv_v2["height calibrated"]
)
plt.xlabel("Year")
plt.ylabel("Greenland Ice Sheet Height (Calibrated, m)")
plt.grid()
plt.show()


In [ ]:
# Parameter definition and optimiser

from scipy.optimize import minimize 
from scipy.interpolate import interp1d


Tm = 0 #fairly certain of this fact
T0 = -10 #degC, roughly!

dT, h = sym.symbols('dT, h')

t0 = t_historical.iloc[0]
t_years = (t_historical - t0).dt.days.values / 365.25

h_data = h_historical.values
h0 = h_data[0]


def delta_T(t):
    return 0.5 + 0.05*t

delta_T_interp = interp1d(
    t_years,
    delta_T(t_years),
    kind='linear',
    fill_value="extrapolate"
)



def dh_dt(t, h,r, F, P ):

    h = max(h[0], 1.0) #enforcing a muin h so that the solver doesnt break with high gradients

    dT = float(delta_T_interp(t))
    T = T0+ dT

    melt = r * (T - Tm)**2 / h
    flow = F * h

    dhdt = P - melt - flow

    return [dhdt]

# Given at some t=0, dh/dt = 0, rearrange equation for P 

def compute_P(r, F, h0, t0):
    dT0 = delta_T(t0)
    return (r * (T0 + dT0 - Tm)**2) / h0 + F*h0

def simulate_height(r, F, P , t_span, h0): # model predicted height

    sol = solve_ivp(
        lambda t, h: dh_dt(t, h, r, F, P),
        t_span=[t_span[0], t_span[-1]],
        y0=[h0],
        t_eval=t_span,
        max_step=0.25
    )
    return sol.y[0]


def objective(params):
    r, F, P = params

    if r < 0 or F < 0 or P < 0:
        return 1e20 #if number non-physical c, apply large penalty

    h_model = simulate_height(r, F,P,   t_years, h0)
    eps = 1e-6
    rel_error = (h_model - h_data) / (h_data + eps)

    return np.sum(rel_error**2)



initial_guess = [1.0, 1e-3, 1.5]  # sensible starting values
bounds = [(0, None), (0, None), (0, None)]  # r ≥ 0, F ≥ 0

result = minimize(objective, initial_guess, bounds=bounds, method='L-BFGS-B') #  minimise the objective function
r_opt, F_opt, P_opt = result.x

h_fit = simulate_height(r_opt, F_opt, P_opt, t_years, h0)

print("Optimised parameters:")
print(f"r = {r_opt}")
print(f"F = {F_opt}")
print(f"P = {P_opt}")

plt.figure(figsize=(10,5))
plt.plot(t_historical, h_data, label="Historical (calibrated)")
plt.plot(t_historical, h_fit, label="Model fit")
plt.xlabel("Year")
plt.ylabel("Ice Sheet Height (m)")
plt.legend()
plt.grid()
plt.show()



# def solve_model(r, F, t_span, h0):
#     P = compute_P(r, F, h0, t_span[0])

#     def dhdt(t, h):
#         h_val = max(h[0], 1.0)
#         dT = delta_T(t)
#         dhdt_val = P - (r * (T0 + dT - Tm)**2)/h_val - F*h_val
#         return [dhdt_val]

#     sol = solve_ivp(
#         dhdt,
#         t_span,
#         [h0],
#         t_eval=t_historical,
#         method='RK45'
#     )

#     if not sol.success:
#         return None
    
#     return sol.y[0]

# def error(params):
#     r, F = params
#     h_model = solve_model(r, F, (t_historical[0], t_historical[-1]), h_historical[0])
#     if h_model is None:
#         return 1e20
#     return np.mean((h_model - h_historical)**2)


# result = minimize(error, x0=[1e4, 0.001], bounds=[(1e2, 1e7), (1e-6, 1)])
# r_opt, F_opt = result.x
# P_opt = compute_P(r_opt, F_opt, h_historical[0], t_historical[0])

In [ ]:


# use optimized parameters
r, F, P = r_opt, F_opt, P_opt

h_vals = np.linspace(10, 2500, 1000)  # h in meters
t_plot = 0  # pick t=0 or any fixed t

dT0 = float(delta_T_interp(t_plot))
T = T0 + dT0

dhdt_vals = P - r*(T - Tm)**2 / np.maximum(h_vals, 1) - F*h_vals

plt.figure(figsize=(8,5))
plt.plot(h_vals, dhdt_vals)
plt.axhline(0, color='r', linestyle='--', label='dh/dt=0 (equilibrium)')
plt.xlabel('Ice Sheet Height h (m)')
plt.ylabel('dh/dt (m/year)')
# plt.title('Greenland:  dh/dt vs h')
plt.grid()
plt.legend()
plt.show()

In [ ]:


dT, h = sym.symbols('dT, h')

# T0 = -1.5 #degC
# Tm = 0 #degC
# P  = 0.3 #m (up to 1.5m apparently)


# F  = 10 # m/year? 
# r = 1e4 # dimensionless?

######### Attempt to make more informed guess on the values! 

h0 = 3000
Tm = 0 #fairly certain of this fact
T0 = -10 #degC, roughly!
P = 30 #m/year # Glacier Greenland, Ice Sheet, Melting | Britannica.... REF

# Thus we must enforce that P - F*h0 must be > 0. Since we have a good guess of P, 
# F < P/h0 

F = 0.8 * (P/h0)
print(f"F = {F}")
print(f"P/h0 = {P/h0}")
print(f"P - F*h0 = {(P - F*h0)}")

r = (h0 * (P - (F*h0))) / ((T0 - Tm)**2)

print(f"r = {r}")


def delta_T(t): 
    return 5 + 0.01*t #linear warming of 0.1degC per year 


def dh_dt(t, h):
    dT = delta_T(t)

    h_safe = np.maximum(h, 1.0) # enforcing a muin h so that the solver doesnt break with high gradients

    dhdt = P - (r * (T0 + dT - Tm)**2)/h_safe - F*h_safe
    dhdt = np.where((h <= 0) & (dhdt < 0), 0, dhdt)
    return dhdt


h_vals = np.linspace(50, 4000, 500)  # avoid h=0 to prevent division
plt.figure()
plt.plot(h_vals, dh_dt(0, h_vals))
plt.axhline(0, linestyle='--')
plt.xlabel('h (m)')
plt.ylabel('dh/dt (m/year)')
plt.grid()
plt.show()


# Compute tipping point 

dh_dt_symb =  P - (( r *(T0 + dT - Tm) **2 )/h) - (F*h)

func = sym.simplify(dh_dt_symb * h)
discr = sym.discriminant(func, h) 
print(f"Distriminant: {discr}")

tipping_points = sym.solve(discr, dT)
print("Tipping points:", tipping_points)





In [ ]:

h_vals = np.linspace(50, 2500, 500)  # reasonable range
deltaTs = [0, 5, 10, 22, 25]  # example temperature increases

plt.figure(figsize=(6,4))
for dT in deltaTs:
    dhdt_vals = P - (r * (T0 + dT - Tm)**2)/h_vals - F*h_vals
    plt.plot(h_vals, dhdt_vals, label=f'ΔT={dT}°C')
plt.axhline(0, linestyle='--', color='k')
plt.xlabel('Ice sheet height h (m)')
plt.ylabel('dh/dt (m/year)')
plt.title('dh/dt vs h for different ΔT')
plt.grid()
plt.legend()
plt.show()

t_upper = 10000
t_span = [0, t_upper]
t_eval = np.linspace(0, t_upper, 500)
h0_1 = 100
h0_2 = 1500

sol1 = solve_ivp(dh_dt, t_span, [h0_1], t_eval=t_eval)
sol2 = solve_ivp(dh_dt, t_span, [h0_2], t_eval=t_eval)

plt.figure(figsize=(6,4))
plt.plot(sol1.t, sol1.y[0], label=f'h0={h0_1} m')
plt.plot(sol2.t, sol2.y[0], label=f'h0={h0_2} m')
plt.xlabel('Time (years)')
plt.ylabel('Ice sheet height h(t) (m)')
plt.title('Ice sheet evolution over time')
plt.grid()
plt.legend()
plt.show()

In [ ]:
t_start = 0
t_end   = 200
t = np.linspace(t_start, t_end, 10)

# h0_1 = 1.5
# h0_2 = 2.5

h0_1 = tipping_points[0] *0.9
h0_2 = tipping_points[1] *1.1
sol1 = solve_ivp(dh_dt, [t_start, t_end], [h0_1], t_eval=t)
sol2 = solve_ivp(dh_dt, [t_start, t_end], [h0_2], t_eval=t)

plt.plot(sol1.t, sol1.y[0], label=f'h0={h0_1} m')
plt.plot(sol2.t, sol2.y[0], label=f'h0={h0_2} m')
plt.plot(sol1.t, delta_T(sol1.t), label=r'$\delta T(t)$')
# plt.hlines(6.25, 0, 200, 'r', '--')
# plt.vlines(125, 0, 7, 'r', '--')

plt.xlabel('Time (years)')
plt.ylabel('Ice sheet height h(t) (m)')

plt.grid()
plt.legend()
plt.show()


In [ ]:
ax = plt.figure().add_subplot(111)

h_val = 1000

hv = np.linspace(0, 2000, 100)
ax.plot(hv, dh_dt(hv, h))

ax.set_xlabel(r'$h$')
ax.set_ylabel(r'$f(x, h)$', rotation=0)
ax.hlines(0, -1.5, 1.5, linestyles='dashed')

for eq in t_cond:
    ax.plot(eq, 0, 'ro')



dh_dt = P - (( r *(T0 + dT - Tm) **2 )/h) - (F*h)

def dh_dt(h, dT):
    return P - (r * (T0 + dT - Tm)**2)/h - F*h


h_vals = np.linspace(1, 2000, 200) 
dT_val = 0.5

plt.figure()
plt.plot(h_vals, dh_dt(h_vals, dT_val))
plt.axhline(0, linestyle='dashed')
plt.xlabel('h')
plt.ylabel('dh/dt')
plt.grid()
plt.show()


func = sym.simplify(dh_dt * h)

discr = sym.discriminant(func, h) 
#this is just expanded form of:
# P^2 -4Fr(T0 + dT -Tm)^2

# Find real roots: 
rl_rts = sym.real_root(discr)

t_cond = sym.solve(discr, dT)
print(f"Equilibira: {t_cond}")


fig = plt.figure()
ax1 = fig.add_subplot(121)
ax2 = fig.add_subplot(122)

# delta_t = 0.1